# 1. Imports et chargement des données

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    classification_report, ConfusionMatrixDisplay
)
from sklearn.decomposition import PCA

In [ ]:
merged = pd.read_csv("../data/merged.csv")
print(merged.columns)

In [ ]:
for col in merged.select_dtypes(include="number").columns:
    print(col, merged[col].mean())

# 2. Préparation des données — variable `higher`

In [ ]:
num_cols = ['age', 'Medu', 'Fedu', 'traveltime', 'studytime',
            'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health']

merged_model = merged[num_cols + ['higher']].dropna()
X_raw = merged_model[num_cols].values
y = (merged_model['higher'] == 'yes').astype(int).values

print(f"{X_raw.shape[0]} individus | classes : {np.bincount(y)} (0=non, 1=oui)")
print(pd.Series(y).value_counts(normalize=True))

In [ ]:
# Split unique, réutilisé pour tous les modèles
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

# 3. KNN — sélection de K et accuracy

In [ ]:
grid = GridSearchCV(
    KNeighborsClassifier(),
    param_grid={"n_neighbors": range(1, 31)},
    scoring="accuracy",
    cv=10
)
grid.fit(X_train, y_train)

K_opt = grid.best_params_["n_neighbors"]
print(f"K optimal : {K_opt} | Accuracy CV : {grid.best_score_:.3f}")

results = grid.cv_results_
ks = np.array(results["param_n_neighbors"].data, dtype=int)

plt.figure(figsize=(8, 4))
plt.errorbar(ks, results["mean_test_score"], yerr=results["std_test_score"],
             fmt="o-", capsize=4, color="steelblue")
plt.axvline(K_opt, color="red", linestyle="--", label=f"K_opt = {K_opt}")
plt.title("Sélection de K par validation croisée (10 plis)")
plt.xlabel("Nombre de voisins K")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
y_pred_knn = grid.best_estimator_.predict(X_test)
acc_test = accuracy_score(y_test, y_pred_knn)
print(f"Accuracy finale sur jeu de test : {acc_test:.3f}")

# 4. Régression logistique — linéaire et quadratique

In [ ]:
lr_linear = LogisticRegression(max_iter=1000, random_state=42)
lr_linear.fit(X_train, y_train)

y_pred_lin = lr_linear.predict(X_test)
print(f"Accuracy RL linéaire : {accuracy_score(y_test, y_pred_lin):.3f}")

In [ ]:
quad_features = PolynomialFeatures(degree=2, include_bias=False)
X_train_quad = quad_features.fit_transform(X_train)
X_test_quad = quad_features.transform(X_test)

lr_quadratic = LogisticRegression(max_iter=2000, random_state=42)
lr_quadratic.fit(X_train_quad, y_train)

y_pred_quad = lr_quadratic.predict(X_test_quad)
print(f"Accuracy RL quadratique : {accuracy_score(y_test, y_pred_quad):.3f}")

In [ ]:
importances = np.abs(lr_linear.coef_[0])

df_importance = pd.DataFrame({
    'Variable': num_cols,
    'Importance': importances
}).sort_values(by='Importance', ascending=True)

plt.figure(figsize=(10, 8))
plt.barh(df_importance['Variable'], df_importance['Importance'], color='steelblue')
plt.title("Importance des variables dans la Régression Logistique")
plt.xlabel("Poids absolu du coefficient (Importance)")
plt.ylabel("Variables")
plt.tight_layout()
plt.show()

# 5. Évaluation — matrices de confusion et métriques par classe

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_knn,
    display_labels=["No higher (0)", "Higher (1)"],
    cmap="Blues", ax=axes[0]
)
axes[0].set_title("KNN")

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_lin,
    display_labels=["No higher (0)", "Higher (1)"],
    cmap="Blues", ax=axes[1]
)
axes[1].set_title("Logistique Linéaire")

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_quad,
    display_labels=["No higher (0)", "Higher (1)"],
    cmap="Blues", ax=axes[2]
)
axes[2].set_title("Logistique Quadratique")

plt.suptitle("Matrices de confusion — higher (yes/no)", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
modeles = {
    "KNN": y_pred_knn,
    "RL lin.": y_pred_lin,
    "RL quad.": y_pred_quad,
}

for nom, y_pred in modeles.items():
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average=None, zero_division=0)
    rec = recall_score(y_test, y_pred, average=None, zero_division=0)
    print(f"{nom:10s} | Acc : {acc:.3f} | "
          f"P. cl. 0 : {prec[0]:.3f} | P. cl. 1 : {prec[1]:.3f} | "
          f"R. cl. 0 : {rec[0]:.3f} | R. cl. 1 : {rec[1]:.3f}")

In [ ]:
print(classification_report(y_test, y_pred_lin,
      target_names=["No higher (0)", "Higher (1)"], zero_division=0))

# 6. Frontières de décision — School × Famsize

In [ ]:
index_school = num_cols.index('age')   # à adapter si school/famsize sont dans num_cols
index_famsize = 4

famsize_mean, famsize_std = scaler.mean_[index_famsize], scaler.scale_[index_famsize]
school_mean, school_std = scaler.mean_[index_school], scaler.scale_[index_school]

x_plot = X_test_raw[:, index_school]
y_plot = X_test_raw[:, index_famsize]

school_vals = np.linspace(x_plot.min() - 1, x_plot.max() + 1, 100)
famsize_vals = np.linspace(y_plot.min() - 1, y_plot.max() + 1, 100)
xx_real, yy_real = np.meshgrid(school_vals, famsize_vals)

grid_scaled = np.zeros((10000, X_train.shape[1]))
grid_scaled[:, index_school] = (xx_real.ravel() - school_mean) / school_std
grid_scaled[:, index_famsize] = (yy_real.ravel() - famsize_mean) / famsize_std

Z_lin = lr_linear.predict(grid_scaled).reshape(xx_real.shape)
Z_quad = lr_quadratic.predict(quad_features.transform(grid_scaled)).reshape(xx_real.shape)

fig, ax = plt.subplots(1, 2, figsize=(14, 6))

x_jitter = x_plot + np.random.uniform(-0.2, 0.2, size=len(x_plot))
y_jitter = y_plot + np.random.uniform(-0.2, 0.2, size=len(y_plot))

ax[0].contourf(xx_real, yy_real, Z_lin, alpha=0.3, cmap='bwr')
ax[0].scatter(x_jitter, y_jitter, c=y_test, edgecolors='k', cmap='bwr', alpha=0.7)
ax[0].set_title("Frontière Linéaire (Projetée sur School × Famsize)")
ax[0].set_xlabel("School (Encodé)")
ax[0].set_ylabel("Family Size (Encodé)")

ax[1].contourf(xx_real, yy_real, Z_quad, alpha=0.3, cmap='bwr')
ax[1].scatter(x_jitter, y_jitter, c=y_test, edgecolors='k', cmap='bwr', alpha=0.7)
ax[1].set_title("Frontière Quadratique (Projetée sur School × Famsize)")
ax[1].set_xlabel("School (Encodé)")
ax[1].set_ylabel("Family Size (Encodé)")

plt.tight_layout()
plt.show()

# 7. Préparation des données — `reussite_mat` et `reussite_por`

In [ ]:
targets = ['reussite_mat', 'reussite_por']

datasets = {}
for cible in targets:
    df_tmp = merged[num_cols + [cible]].dropna()
    X_raw_c = df_tmp[num_cols].values
    y_tmp = df_tmp[cible].astype(int).values

    sc = StandardScaler()
    X_sc = sc.fit_transform(X_raw_c)

    X_tr, X_te, y_tr, y_te = train_test_split(
        X_sc, y_tmp, test_size=0.2, random_state=42, stratify=y_tmp
    )
    datasets[cible] = dict(X_train=X_tr, X_test=X_te,
                            y_train=y_tr, y_test=y_te, scaler=sc)
    print(f"{cible} | n={len(y_tmp)} | classes {np.bincount(y_tmp)} (0=échec, 1=réussite)")

# 8. KNN et Régression Logistique Quadratique — `reussite_mat` / `reussite_por`

In [ ]:
quad = PolynomialFeatures(degree=2, include_bias=False)

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
fig.suptitle("KNN & Régression Logistique Quadratique — reussite_mat vs reussite_por",
              fontsize=14, y=1.01)

for row_idx, cible in enumerate(targets):
    d = datasets[cible]
    X_tr, X_te = d['X_train'], d['X_test']
    y_tr, y_te = d['y_train'], d['y_test']

    grid_knn = GridSearchCV(
        KNeighborsClassifier(),
        param_grid={"n_neighbors": range(1, 31)},
        scoring="accuracy", cv=10
    )
    grid_knn.fit(X_tr, y_tr)
    K_opt = grid_knn.best_params_["n_neighbors"]
    y_pred_knn = grid_knn.best_estimator_.predict(X_te)

    X_tr_quad = quad.fit_transform(X_tr)
    X_te_quad = quad.transform(X_te)

    lr_quad = LogisticRegression(max_iter=2000, random_state=42)
    lr_quad.fit(X_tr_quad, y_tr)
    y_pred_quad_r = lr_quad.predict(X_te_quad)

    print(f"  {cible.upper()}  (seuil = {seuil})")
    print(f"  KNN  K={K_opt:2d}  | Accuracy : {accuracy_score(y_te, y_pred_knn):.3f}")
    print(classification_report(y_te, y_pred_knn,
          target_names=["Échec (0)", "Réussite (1)"], zero_division=0))
    print(f"  Logistique Quad | Accuracy : {accuracy_score(y_te, y_pred_quad_r):.3f}")
    print(classification_report(y_te, y_pred_quad_r,
          target_names=["Échec (0)", "Réussite (1)"], zero_division=0))

    ks = np.array(grid_knn.cv_results_["param_n_neighbors"].data, dtype=int)
    axK = axes[row_idx][0]
    axK.errorbar(ks, grid_knn.cv_results_["mean_test_score"],
                  yerr=grid_knn.cv_results_["std_test_score"],
                  fmt="o-", capsize=3, color="steelblue", markersize=4)
    axK.axvline(K_opt, color="red", linestyle="--", label=f"K={K_opt}")
    axK.set_title(f"{cible}\nSélection K (CV 10 plis)")
    axK.set_xlabel("K")
    axK.set_ylabel("Accuracy")
    axK.legend()

    axI = axes[row_idx][1]
    feat_names = quad.get_feature_names_out(num_cols)
    imp = pd.Series(np.abs(lr_quad.coef_[0]), index=feat_names).nlargest(15).sort_values()
    imp.plot(kind="barh", ax=axI, color="steelblue")
    axI.set_title(f"{cible}\nTop 15 variables (LR Quad)")
    axI.set_xlabel("Coeff. absolu")

    ConfusionMatrixDisplay.from_predictions(
        y_te, y_pred_knn,
        display_labels=["Échec (0)", "Réussite (1)"],
        cmap="Blues", ax=axes[row_idx][2]
    )
    axes[row_idx][2].set_title(f"{cible}\nConfusion KNN (K={K_opt})")

    ConfusionMatrixDisplay.from_predictions(
        y_te, y_pred_quad_r,
        display_labels=["Échec (0)", "Réussite (1)"],
        cmap="Oranges", ax=axes[row_idx][3]
    )
    axes[row_idx][3].set_title(f"{cible}\nConfusion LR Quadratique")

plt.tight_layout()
plt.show()

# 9. Frontières de décision — meilleurs axes par cible

In [ ]:
best_axes = {
    'reussite_mat': (0, 1),
    'reussite_por': (0, 1),
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Frontières de décision (meilleurs axes) — projection 2D", fontsize=14)

for row_idx, cible in enumerate(targets):
    d = datasets[cible]
    X_tr, X_te = d['X_train'], d['X_test']
    y_tr, y_te = d['y_train'], d['y_test']

    IDX0, IDX1 = best_axes[cible]
    label0, label1 = num_cols[IDX0], num_cols[IDX1]

    X_tr_2d = X_tr[:, [IDX0, IDX1]]
    X_te_2d = X_te[:, [IDX0, IDX1]]

    grid_knn_2d = GridSearchCV(
        KNeighborsClassifier(),
        param_grid={"n_neighbors": range(1, 31)},
        scoring="accuracy", cv=10
    )
    grid_knn_2d.fit(X_tr_2d, y_tr)
    K_opt_2d = grid_knn_2d.best_params_["n_neighbors"]
    knn_2d = grid_knn_2d.best_estimator_

    quad_2d = PolynomialFeatures(degree=2, include_bias=False)
    X_tr_q2d = quad_2d.fit_transform(X_tr_2d)
    lr_2d = LogisticRegression(max_iter=2000, random_state=42)
    lr_2d.fit(X_tr_q2d, y_tr)

    x_min, x_max = X_te_2d[:, 0].min() - 0.5, X_te_2d[:, 0].max() + 0.5
    y_min, y_max = X_te_2d[:, 1].min() - 0.5, X_te_2d[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                          np.linspace(y_min, y_max, 300))
    grid_2d = np.c_[xx.ravel(), yy.ravel()]

    Z_knn = knn_2d.predict(grid_2d).reshape(xx.shape)
    Z_quad_2d = lr_2d.predict(quad_2d.transform(grid_2d)).reshape(xx.shape)

    colors_bg = ["#d6e8f7", "#f7d6d6"]

    for col_idx, (Z, titre) in enumerate([
        (Z_knn, f"KNN (K={K_opt_2d})"),
        (Z_quad_2d, "LR Quadratique"),
    ]):
        ax = axes[row_idx][col_idx]

        ax.contourf(xx, yy, Z, alpha=0.35, levels=[-0.5, 0.5, 1.5], colors=colors_bg)
        ax.contour(xx, yy, Z, levels=[0.5], colors="black", linewidths=1.2, linestyles="--")

        jitter = np.random.default_rng(0).uniform(-0.05, 0.05, X_te_2d.shape)
        for cls, col, lbl in [(0, "steelblue", "Échec"), (1, "tomato", "Réussite")]:
            mask = y_te == cls
            ax.scatter(X_te_2d[mask, 0] + jitter[mask, 0],
                       X_te_2d[mask, 1] + jitter[mask, 1],
                       c=col, edgecolors="k", linewidths=0.4,
                       s=40, alpha=0.8, label=lbl)

        ax.set_title(f"{cible}\n{titre}")
        ax.set_xlabel(f"{label0} (normalisé)")
        ax.set_ylabel(f"{label1} (normalisé)")
        ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

# 10. Frontières de décision — exploration de toutes les paires de variables

In [ ]:
from itertools import combinations

all_combos = list(combinations(range(len(num_cols)), 2))[:40]

fig, axes = plt.subplots(len(all_combos), 4, figsize=(24, 4 * len(all_combos)))

for row, (IDX0, IDX1) in enumerate(all_combos):
    label0 = num_cols[IDX0]
    label1 = num_cols[IDX1]

    for col, (cible, modele_nom) in enumerate([
        ("reussite_mat", "KNN"),
        ("reussite_mat", "RL²"),
        ("reussite_por", "KNN"),
        ("reussite_por", "RL²")
    ]):
        d = datasets[cible]
        X_tr = d["X_train"][:, [IDX0, IDX1]]
        X_te = d["X_test"][:, [IDX0, IDX1]]
        y_tr = d["y_train"]
        y_te = d["y_test"]

        ax = axes[row, col]

        if modele_nom == "KNN":
            model = KNeighborsClassifier(n_neighbors=15)
            model.fit(X_tr, y_tr)
            predictor = lambda X: model.predict(X)
        else:
            poly = PolynomialFeatures(degree=2, include_bias=False)
            X_tr_poly = poly.fit_transform(X_tr)
            model = LogisticRegression(max_iter=5000)
            model.fit(X_tr_poly, y_tr)
            predictor = lambda X: model.predict(poly.transform(X))

        x_min, x_max = X_te[:, 0].min() - 0.5, X_te[:, 0].max() + 0.5
        y_min, y_max = X_te[:, 1].min() - 0.5, X_te[:, 1].max() + 0.5

        xx, yy = np.meshgrid(np.linspace(x_min, x_max, 150),
                              np.linspace(y_min, y_max, 150))

        grid_pts = np.c_[xx.ravel(), yy.ravel()]
        Z = predictor(grid_pts).reshape(xx.shape)

        ax.contourf(xx, yy, Z, alpha=0.35, levels=[-0.5, 0.5, 1.5])
        ax.contour(xx, yy, Z, levels=[0.5], colors="black", linewidths=1)

        ax.scatter(X_te[y_te == 0, 0], X_te[y_te == 0, 1], s=15, alpha=0.7, label="0")
        ax.scatter(X_te[y_te == 1, 0], X_te[y_te == 1, 1], s=15, alpha=0.7, label="1")

        if row == 0:
            ax.set_title(f"{cible.replace('reussite_', '').upper()} - {modele_nom}")

        if col == 0:
            ax.set_ylabel(f"{label0}\n{label1}", fontsize=9)

        ax.set_xticks([])
        ax.set_yticks([])

plt.tight_layout()
plt.show()